# Lab 06: Agentic RAG from scratch

Build a research-style RAG agent against a *controlled* corpus of bundled
Markdown documents — the corpus-you-control counterpart to Lab 03's
search-the-open-web agent. Two tools, one loop, citation tracking at
chunk granularity. From scratch in pure Python — no LangChain, no
vector-store SDK, no embeddings API by default.

This is the runnable companion to
[`labs/06-agentic-rag-from-scratch/README.md`](./README.md). Read the
brief first.

**Estimated time:** 100–130 minutes.
**Difficulty:** 🟡 Intermediate.
**Prerequisites:** Labs 01, 02, 03, plus the three Path 02 concept pages
([what-is-rag](../../concepts/rag/what-is-rag.md),
[retrieval-as-a-tool](../../concepts/rag/retrieval-as-a-tool.md),
[chunking-and-indexing](../../concepts/rag/chunking-and-indexing.md)).

> 🔴 **Embedding libraries are fast-changing.** This notebook is pinned
> to `sentence-transformers>=5.0,<6.0` (verified 2026-05-24, latest
> `5.5.1`). If something doesn't behave as described, check the
> [embedding-models snapshot](../../tools/embeddings/snapshot-v1.0.md)
> first — the library's API has been stable but the model leaderboard
> moves quarterly.

## Step 0: Setup

Same provider-agnostic LLM client pattern as prior labs. Two new
dependencies for this lab: `sentence-transformers` for the local
embedding model, `numpy` (already installed) for the index math.

```bash
uv add 'sentence-transformers>=5.0,<6.0'
```

The first call to `SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")`
will download the model (~80 MB) on first run. Add 30-60 seconds on
the first execution; subsequent runs load from cache.

In [ ]:
import os
import pathlib
import json
import hashlib
import re
from typing import Any
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = "openai"   # or "anthropic"

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)
print(f"Provider: {PROVIDER}")


**Sample output:**

```
Provider: openai
```

In [ ]:
# Provider-agnostic chat client — same wrapper used since Lab 01.

def chat_with_tools(messages: list[dict], tools: list[dict],
                    model: str | None = None) -> dict:
    """Call chat completion with tool schemas. Return assistant message."""
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            messages=messages,
            tools=tools,
            temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                }
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anthropic_tools = [
            {
                "name": t["function"]["name"],
                "description": t["function"]["description"],
                "input_schema": t["function"]["parameters"],
            }
            for t in tools
        ]
        system = next((m["content"] for m in messages if m["role"] == "system"),
                      None)
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            system=system or "",
            messages=non_system,
            tools=anthropic_tools,
            max_tokens=2048,
        )
        content_blocks = resp.content
        text = "".join(b.text for b in content_blocks if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in content_blocks
            if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    else:
        raise ValueError(f"Unknown provider: {PROVIDER}")


print("LLM client ready.")


## Step 1: Load and chunk the corpus

Eight Markdown documents in `./corpus/`. We split each into chunks of
roughly 160 tokens with ~20% overlap, attaching metadata
(`doc_id`, `chunk_id`, `title`). The 160-token target is deliberate:
`all-MiniLM-L6-v2` silently truncates inputs over 256 *wordpieces*
(roughly ~200 LLM-tokens). The chunker's overlap-prepending step adds
~32 tokens to most chunks, so a 160-token target keeps us comfortably
under the 200-token effective cap. Going over the limit doesn't error
— it just discards content. See
[`chunking-and-indexing.md`](../../concepts/rag/chunking-and-indexing.md)
for the full discussion of the 256-wordpiece foot-gun.

The chunker uses *recursive splitting on document structure*:
prefer to split on paragraph breaks (`\n\n`), fall back to sentence
breaks (`. `), fall back to word breaks only as a last resort. This
preserves natural unit boundaries so embeddings represent coherent
ideas.

In [ ]:
# Chunker — recursive splitting with overlap
# Target: ~200 tokens/chunk, ~40 token overlap (~20%)
# Tokens are approximated as words / 0.75 (English ratio)

CORPUS_DIR = pathlib.Path("./corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    """Word count divided by ~0.75 — close enough for chunking decisions."""
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    """Split on blank lines."""
    parts = re.split(r"\n\s*\n", text)
    return [p.strip() for p in parts if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    """Split on sentence-ending punctuation followed by whitespace."""
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]


def chunk_text(text: str, target_tokens: int = TARGET_TOKENS,
               overlap_tokens: int = OVERLAP_TOKENS) -> list[str]:
    """Recursive splitter: paragraph → sentence → word, target ~N tokens."""
    paragraphs = split_at_paragraphs(text)

    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = approx_tokens(para)

        # Paragraph too big — recurse into sentences
        if para_tokens > target_tokens:
            # Flush current accumulator
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0

            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > target_tokens and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue

        # Paragraph fits — accumulate or flush
        if current_tokens + para_tokens > target_tokens and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens

    if current:
        chunks.append("\n\n".join(current))

    # Add overlap by prepending the tail of the previous chunk to each chunk
    if overlap_tokens <= 0 or len(chunks) < 2:
        return chunks

    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        # Number of words ~= tokens * 0.75
        overlap_words = int(overlap_tokens * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip()
                          if tail else chunks[i])
    return overlapped


# Load all docs
all_chunks = []

def first_heading(text: str) -> str:
    """Return the first '# Heading' line, or '' if none."""
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return ""


for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    title = first_heading(text)
    chunks = chunk_text(text)
    for i, chunk_body in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "title": title,
            "text": chunk_body,
            "tokens_est": approx_tokens(chunk_body),
        })

print(f"Loaded {len({c['doc_id'] for c in all_chunks})} documents")
print(f"Split into {len(all_chunks)} chunks")
print()
print("Chunk size distribution (estimated tokens):")
import statistics
sizes = [c["tokens_est"] for c in all_chunks]
print(f"  min={min(sizes)}, max={max(sizes)}, "
      f"median={int(statistics.median(sizes))}, "
      f"mean={int(statistics.mean(sizes))}")
print()
print("First few chunks:")
for c in all_chunks[:3]:
    snippet = c["text"][:100].replace("\n", " ")
    print(f"  [{c['chunk_id']}] ({c['tokens_est']} tok) {snippet}...")


**Sample output (yours will vary by chunker output):**

```
Loaded 8 documents
Split into 24 chunks

Chunk size distribution (estimated tokens):
  min=58, max=212, median=185, mean=174

First few chunks:
  [01-agent-loop.md:0] (188 tok) # The Agent Loop: A Brief Introduction  The agent loop is the core control...
  [01-agent-loop.md:1] (165 tok) is the user's question, the previous turn's output, or the result of...
  [01-agent-loop.md:2] (147 tok) In the **observe** phase, the tool's result gets appended to the conversation...
```

Note that the chunks have ~20% overlap — each chunk's first few words
are the *tail* of the previous chunk. This recovers facts that would
otherwise straddle a chunk boundary. The cost is ~20% more storage and
embedding compute, which is essentially free at our scale.

The chunker is ~50 lines. Production RAG systems often use LangChain's
`RecursiveCharacterTextSplitter` which is more sophisticated, but the
algorithm is the same: prefer natural boundaries, recurse if a unit is
too big, accumulate until you hit the target size.

## Step 2: Build the embedding index

Load `all-MiniLM-L6-v2`, encode all chunks with
`normalize_embeddings=True` (so cosine similarity is exactly a dot
product downstream), and stack into a single numpy array of shape
`(n_chunks, 384)`. That array *is* the index.

`normalize_embeddings=True` is non-optional for our cosine math: if
embeddings have unit norm, then `cos(a, b) = a · b`. Without
normalization, you have to divide by `‖a‖‖b‖` in every similarity
computation. Forget the flag and your similarity scores will look
nothing like you expect.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# First run downloads the model (~80 MB); subsequent runs use cache
print("Loading embedding model (downloads on first run; ~80 MB)...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)

# Encode all chunk texts. shape: (n_chunks, 384)
chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,   # crucial — cosine = dot product
    convert_to_numpy=True,        # default
    show_progress_bar=False,
)

print(f"Embeddings: shape={embeddings.shape}, dtype={embeddings.dtype}")
print(f"Memory: {embeddings.nbytes / 1024:.1f} KB")
print()

# Sanity: L2 norm of each row should be ~1.0 if normalized correctly
norms = np.linalg.norm(embeddings, axis=1)
print(f"Norms: min={norms.min():.4f}, max={norms.max():.4f}, "
      f"mean={norms.mean():.4f} (should all be ~1.0)")


**Sample output:**

```
Loading embedding model (downloads on first run; ~80 MB)...
Embeddings: shape=(24, 384), dtype=float32
Memory: 36.0 KB

Norms: min=1.0000, max=1.0000, mean=1.0000 (should all be ~1.0)
```

That's a real vector index. 24 chunks × 384 dims × 4 bytes = 36 KB of
floats. The "index" is the array plus the parallel chunk metadata
list (`all_chunks`). A production vector store like Chroma or Qdrant
wraps the same math with persistence, network APIs, and ANN
algorithms for billions-of-vectors scale — but the math is *exactly*
this.

## Step 3: The `search_corpus` tool

The first tool. Embeds the query, computes dot product against every
chunk's embedding (= cosine similarity on normalized vectors), and
returns the top-k chunks with snippets.

Contract:

```python
search_corpus(query: str, top_k: int = 5) -> dict
```

Returns one of:

```python
{"status": "ok",    "results": [{"chunk_id", "doc_id", "title", "snippet", "score"}, ...]}
{"status": "empty", "query": ..., "detail": "no chunks above similarity threshold"}
{"status": "error", "kind": "...", "detail": ...}
```

Same structured-error pattern as Lab 02 and Lab 03 — the agent sees a
shaped result, not a crash.

In [ ]:
# Similarity threshold below which we treat a result as "empty"
# 0.30 is a reasonable cosine cutoff for MiniLM on out-of-corpus queries.
# Tune per corpus; too tight and you lose recall, too loose and you ship noise.
MIN_SIMILARITY = 0.30


def search_corpus(query: str, top_k: int = 5) -> dict:
    """Embed the query, return top-k chunks by cosine similarity."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other",
                "detail": "empty query"}

    try:
        query_emb = embedder.encode(
            [query.strip()],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]
    except Exception as e:
        return {"status": "error", "kind": "embed_failure",
                "detail": f"{type(e).__name__}: {e}"}

    # Dot product against normalized embeddings = cosine similarity
    scores = embeddings @ query_emb  # shape (n_chunks,)

    # Top-k indices, sorted by descending similarity
    n = min(top_k, len(scores))
    top_indices = np.argsort(scores)[::-1][:n]
    top_scores = scores[top_indices]

    # Apply similarity floor — drop results that are clearly off-topic
    above_floor = top_scores >= MIN_SIMILARITY
    if not above_floor.any():
        return {
            "status": "empty",
            "query": query,
            "detail": f"no chunks crossed similarity floor "
                     f"of {MIN_SIMILARITY} (top score was {top_scores.max():.3f})",
        }

    results = []
    for idx, score in zip(top_indices[above_floor],
                          top_scores[above_floor], strict=True):
        chunk = all_chunks[idx]
        # Snippet = first 200 chars of the chunk text
        snippet = chunk["text"][:200].replace("\n", " ")
        if len(chunk["text"]) > 200:
            snippet += "..."
        results.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "snippet": snippet,
            "score": float(score),
        })

    return {"status": "ok", "results": results}


# Smoke test
print("Smoke test — query 'agent loop four phases':")
result = search_corpus("agent loop four phases", top_k=3)
print(f"  status: {result['status']}")
if result["status"] == "ok":
    for r in result["results"]:
        print(f"  [{r['chunk_id']}] score={r['score']:.3f} | {r['title']}")
        print(f"    snippet: {r['snippet'][:120]}...")


**Sample output (yours will vary slightly):**

```
Smoke test — query 'agent loop four phases':
  status: ok
  [01-agent-loop.md:0] score=0.711 | The Agent Loop: A Brief Introduction
    snippet: # The Agent Loop: A Brief Introduction  The agent loop is the core control...
  [01-agent-loop.md:1] score=0.643 | The Agent Loop: A Brief Introduction
    snippet: is the user's question, the previous turn's output, or the result of a...
  [03-react-pattern.md:0] score=0.428 | The ReAct Pattern
    snippet: # The ReAct Pattern  ReAct is the now-canonical way of structuring an agent...
```

The retrieval works: a query about the agent loop's phases returns the
two chunks of `01-agent-loop.md` (the relevant document) with high
scores, plus a tangentially related chunk from `03-react-pattern.md`
at a lower score. The model can read the snippets and decide whether
the snippet alone answers the question, or whether to fetch the full
chunk via `read_chunk`.

## Step 4: The `read_chunk` tool

The second tool. Returns the full text of a single chunk by `chunk_id`.
Trivial implementation since chunks are in memory — but the structured
error pattern is still important so the agent can handle a malformed
ID gracefully.

In [ ]:
# Build a lookup dict for O(1) chunk retrieval
chunks_by_id = {c["chunk_id"]: c for c in all_chunks}


def read_chunk(chunk_id: str) -> dict:
    """Return the full text of a single chunk by ID."""
    if not chunk_id:
        return {"status": "error", "kind": "other",
                "detail": "empty chunk_id"}

    chunk = chunks_by_id.get(chunk_id)
    if chunk is None:
        return {
            "status": "error",
            "kind": "not_found",
            "detail": f"no chunk with id {chunk_id!r}; "
                     f"valid IDs come from search_corpus results",
        }

    return {
        "status": "ok",
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "title": chunk["title"],
        "text": chunk["text"],
    }


# Smoke test
print("Smoke test — read first chunk:")
result = read_chunk(all_chunks[0]["chunk_id"])
print(f"  status: {result['status']}")
print(f"  title: {result['title']}")
print(f"  text length: {len(result['text'])} chars")
print()
print("Smoke test — read nonexistent chunk:")
result = read_chunk("nonexistent.md:99")
print(f"  status: {result['status']}, kind: {result['kind']}")
print(f"  detail: {result['detail']}")


**Sample output:**

```
Smoke test — read first chunk:
  status: ok
  title: The Agent Loop: A Brief Introduction
  text length: 1043 chars

Smoke test — read nonexistent chunk:
  status: error, kind: not_found
  detail: no chunk with id 'nonexistent.md:99'; valid IDs come from search_corpus results
```

Note the failure mode is *structured* — the agent gets a typed
`kind: "not_found"` and a helpful `detail` explaining the contract. If
the agent hallucinates a chunk ID, this is what it sees. It can then
re-issue `search_corpus` to get a real ID.

## Step 5: Tool schemas + agent loop

Same shape as Lab 03's agent loop. Two changes:

1. Citation entries are `{chunk_id, doc_id, title}` — chunk-level
   provenance instead of URL-level.
2. The system prompt mentions "the corpus" instead of "the web."

Everything else is identical: `_action_hash` for repeated-action
detection, `MAX_STEPS` cap, structured-error handling, the loop tracks
citations (not the LLM).

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_corpus",
            "description": (
                "Search the corpus by semantic similarity to a query. Returns "
                "the top-k chunks with title, snippet, and similarity score. "
                "Use this to triage which chunks are worth reading in full. "
                "Phrase queries as 3-8 specific words; broader queries return "
                "less relevant results."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query, 3-8 specific words best.",
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "How many results to return. 1-10, default 5.",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_chunk",
            "description": (
                "Read the full text of a single chunk. Use after search_corpus "
                "to inspect a candidate chunk's full content. Pass the chunk_id "
                "exactly as it appeared in a search_corpus result."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string",
                        "description": "Chunk ID from search_corpus results, "
                                       "e.g., '01-agent-loop.md:2'.",
                    },
                },
                "required": ["chunk_id"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> dict:
    """Dispatch a tool call to its implementation."""
    if name == "search_corpus":
        return search_corpus(
            query=args["query"],
            top_k=args.get("top_k", 5),
        )
    if name == "read_chunk":
        return read_chunk(chunk_id=args["chunk_id"])
    return {"status": "error", "kind": "other",
            "detail": f"unknown tool: {name}"}


MAX_STEPS = 8

SYSTEM_PROMPT = """You are a research assistant grounded in a specific document
corpus. Answer the user's question only from the corpus.

Strategy:
1. Start with search_corpus using 3-8 specific words from the question.
2. Look at the snippets and similarity scores. If the snippets alone answer
   the question, synthesize directly. If not, pick the 1-2 most relevant
   chunks and call read_chunk to see their full text.
3. If results are empty or poor, refine the query and search again — but do
   not repeat queries you've already tried.
4. Stop when you can answer confidently with grounded evidence from chunks
   you actually read. Do not loop.
5. In your final answer, cite the chunks you actually read. The system will
   verify this against the read_chunk call log.

When you cannot find a confident answer in the corpus, say so plainly.
"I could not find this in the corpus" is an acceptable response. Do not
guess or use external knowledge.
"""


def _action_hash(name: str, args: dict) -> str:
    """Deterministic hash of a (tool_name, args) pair for dedup."""
    payload = name + "|" + json.dumps(args, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def run_agent(question: str, max_steps: int = MAX_STEPS,
              verbose: bool = True) -> dict:
    """Run the RAG agent. Return answer + chunk-level citations."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")

        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {
                    "id": tc["id"],
                    "type": "function",
                    "function": {"name": tc["name"], "arguments": tc["arguments"]},
                }
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)

        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {msg['content'][:140]}...")
            return {
                "answer": msg["content"],
                "citations": citations,
                "steps": step,
                "stopped_reason": "answer_with_citations" if citations
                else "answer_without_read",
            }

        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)

            if ah in seen_actions:
                tool_result = {
                    "status": "error",
                    "kind": "repeated_action",
                    "detail": (f"You already called {tc['name']} with these "
                              f"arguments. Try different ones."),
                }
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED — refused]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    status = tool_result.get("status", "?")
                    args_repr = (str(args)[:80] + "...") if len(str(args)) > 80 \
                        else str(args)
                    print(f"  → {tc['name']}({args_repr}) → {status}")

                # Citation tracking: only chunks actually read via read_chunk
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append({
                        "chunk_id": tool_result["chunk_id"],
                        "doc_id": tool_result["doc_id"],
                        "title": tool_result["title"],
                    })

            messages.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": json.dumps(tool_result)[:4000],
            })

    if verbose:
        print(f"\n  ⚠ Hit step cap ({max_steps}) without final answer")
    return {
        "answer": (
            "I reached the step limit without a confident answer. "
            f"I read {len(citations)} chunk(s) from the corpus."
        ),
        "citations": citations,
        "steps": max_steps,
        "stopped_reason": "step_cap",
    }


print("Agent ready. Call run_agent('your question here').")


**Sample output:**

```
Agent ready. Call run_agent('your question here').
```

Two important design properties baked into this loop:

- **Citations come from the *loop*, not the LLM.** The moment
  `read_chunk` returns `status: ok`, the loop appends to
  `citations`. The model can't fake a citation it didn't earn by
  actually reading the chunk. This is the Lab 03 pattern, transferred
  to chunk-level provenance unchanged.

- **`_action_hash` dedupes on `(tool_name, args)`.** Two
  `search_corpus` calls with different `top_k` values are different
  actions. Two `read_chunk` calls on the same `chunk_id` are a
  repeat. The mechanism doesn't care what the tools do; it's
  structural.

## Step 6: Three test queries

Easy / medium / hard, in increasing trajectory complexity. Run them
one at a time so the agent's reasoning is readable.

In [ ]:
# EASY: one search, possibly one read, single document.
easy_q = "What is the ReAct pattern in agent design?"
print(f"QUERY: {easy_q}")
print("=" * 70)
result_easy = run_agent(easy_q, verbose=True)
print("=" * 70)
print(f"\n✓ Steps: {result_easy['steps']}, "
      f"Citations: {len(result_easy['citations'])}")
print(f"\nAnswer:\n{result_easy['answer']}")
print("\nCitations:")
for c in result_easy["citations"]:
    print(f"  - [{c['chunk_id']}] {c['title']}")


**Sample output (yours will vary):**

```
QUERY: What is the ReAct pattern in agent design?
======================================================================

── Step 1 ──
  → search_corpus({'query': 'ReAct pattern agent design'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '03-react-pattern.md:0'}) → ok
  ◆ FINAL: The ReAct pattern is a way of structuring an agent's...
======================================================================

✓ Steps: 2, Citations: 1

Answer:
The ReAct pattern (Reason + Act) is a way of structuring an agent's
decision step so that the model emits an explicit "thought" trace
before each action. This makes the agent's reasoning legible and
auditable: you can read the trace and see why the agent chose to call
the tool it did...

Citations:
  - [03-react-pattern.md:0] The ReAct Pattern
```

The trajectory is the expected shape: search → see strong match →
read the chunk → synthesize. One citation because one chunk was
read.

In [ ]:
# MEDIUM: synthesis across two documents.
# Chunking interacts with embedding token limits — info lives in two docs.
medium_q = (
    "How does chunk size interact with the embedding model's token limit, "
    "and what happens if you chunk too aggressively?"
)
print(f"QUERY: {medium_q}")
print("=" * 70)
result_med = run_agent(medium_q, verbose=True)
print("=" * 70)
print(f"\n✓ Steps: {result_med['steps']}, "
      f"Citations: {len(result_med['citations'])}")
print(f"\nAnswer:\n{result_med['answer']}")
print("\nCitations:")
for c in result_med["citations"]:
    print(f"  - [{c['chunk_id']}] {c['title']}")


**Sample output (yours will vary):**

```
QUERY: How does chunk size interact with the embedding model's token limit...
======================================================================

── Step 1 ──
  → search_corpus({'query': 'chunk size token limit embedding'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '07-chunking-strategies.md:1'}) → ok

── Step 3 ──
  → read_chunk({'chunk_id': '05-embeddings.md:2'}) → ok
  ◆ FINAL: Chunk size interacts with embedding token limits in...
======================================================================

✓ Steps: 3, Citations: 2

Answer:
Chunk size interacts with embedding token limits in two ways. First,
every embedding model has a maximum sequence length (256 wordpieces
for all-MiniLM-L6-v2, ~8K tokens for text-embedding-3-small) above
which input gets silently truncated...

Citations:
  - [07-chunking-strategies.md:1] Chunking Strategies for RAG
  - [05-embeddings.md:2] Embeddings: What They Are and How They Behave
```

The trajectory now reaches across two documents — chunking strategies
and the embedding-behavior doc. Two citations because two chunks were
read. The synthesis combines both.

In [ ]:
# HARD: multi-hop synthesis across 3+ chunks.
hard_q = (
    "What's the difference between using search as a tool versus using "
    "retrieval as a tool, and what failure modes does each have?"
)
print(f"QUERY: {hard_q}")
print("=" * 70)
result_hard = run_agent(hard_q, verbose=True)
print("=" * 70)
print(f"\n✓ Steps: {result_hard['steps']}, "
      f"Citations: {len(result_hard['citations'])}")
print(f"\nAnswer:\n{result_hard['answer']}")
print("\nCitations:")
for c in result_hard["citations"]:
    print(f"  - [{c['chunk_id']}] {c['title']}")


**Sample output (yours will vary):**

```
QUERY: What's the difference between using search as a tool versus...
======================================================================

── Step 1 ──
  → search_corpus({'query': 'search tool retrieval tool failure modes'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:0'}) → ok

── Step 3 ──
  → search_corpus({'query': 'tool design structured errors patterns'}) → ok

── Step 4 ──
  → read_chunk({'chunk_id': '02-tool-design.md:1'}) → ok

── Step 5 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:1'}) → ok
  ◆ FINAL: Search and retrieval are two different patterns that share...
======================================================================

✓ Steps: 5, Citations: 3
```

Five steps, three citations. The agent recognized that one search
wasn't enough — refined to a different query, read another chunk,
synthesized across three sources. This is the multi-step pattern the
lab is meant to surface.

## Step 7: Failure-mode walkthrough

Easy questions don't exercise the agent's intelligence. The
failure-recovery patterns do. The corpus is bounded and you own it,
so the failure modes are different from Lab 03's web-world — but
they're just as real.

In [ ]:
# Failure mode 1: EMPTY RESULTS — query about something the corpus doesn't cover

print("─" * 70)
print("FAILURE MODE 1: Query about an off-corpus topic")
print("─" * 70)
# Direct tool test
empty_result = search_corpus("French Revolution 1789 Bastille", top_k=5)
print(f"\nDirect tool call: {empty_result['status']}")
if empty_result["status"] == "empty":
    print(f"  detail: {empty_result['detail']}")
elif empty_result["status"] == "ok":
    # MiniLM does return *something* for any query; the floor filters
    print(f"  → results returned: {len(empty_result['results'])}")
    for r in empty_result["results"][:3]:
        print(f"     score={r['score']:.3f} [{r['chunk_id']}]")

print()
print("Agent behavior on the off-corpus question:")
result_off = run_agent(
    "What started the French Revolution in 1789?",
    verbose=True,
)
print(f"\n  Stopped reason: {result_off['stopped_reason']}")
print(f"  Citations: {len(result_off['citations'])}")
print(f"  Answer (first 200 chars): {result_off['answer'][:200]}")


**Sample output (yours will vary):**

```
──────────────────────────────────────────────────────────────────────
FAILURE MODE 1: Query about an off-corpus topic
──────────────────────────────────────────────────────────────────────

Direct tool call: empty
  detail: no chunks crossed similarity floor of 0.3 (top score was 0.142)

Agent behavior on the off-corpus question:

── Step 1 ──
  → search_corpus({'query': 'French Revolution 1789 Bastille'}) → empty

── Step 2 ──
  → search_corpus({'query': 'history France 18th century'}) → empty

── Step 3 ──
  ◆ FINAL: I could not find any information about the French Revolution...

  Stopped reason: answer_without_read
  Citations: 0
  Answer (first 200 chars): I could not find any information about the French
  Revolution in the corpus. The corpus covers topics related to AI
  agent architecture and retrieval-augmented generation, not historical
  events. I cannot answer this question from the available material.
```

The agent searched, got empty twice, and *correctly surfaced the
absence* without hallucinating from world knowledge. Zero citations
because zero chunks were read — and that's the correct outcome.


In [ ]:
# Failure mode 2: SIMILAR-BUT-WRONG retrieval
# A query whose top match is high-similarity but doesn't actually answer it.

print("─" * 70)
print("FAILURE MODE 2: Similar-but-not-quite-right retrieval")
print("─" * 70)

# This query asks about something specific (chunking 'overlap') but
# similarity may rank a chunk-strategies overview chunk above the
# overlap-specific chunk. The agent should READ and notice the gap.
result_mismatch = run_agent(
    "Specifically, how much overlap should chunks have, and why?",
    verbose=True,
)
print(f"\n  Citations: {len(result_mismatch['citations'])}")
print(f"  Read chunks: {[c['chunk_id'] for c in result_mismatch['citations']]}")
print(f"  Answer (first 300 chars): {result_mismatch['answer'][:300]}")


**Sample output (yours will vary):**

```
──────────────────────────────────────────────────────────────────────
FAILURE MODE 2: Similar-but-not-quite-right retrieval
──────────────────────────────────────────────────────────────────────

── Step 1 ──
  → search_corpus({'query': 'chunk overlap size recommendation'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '07-chunking-strategies.md:0'}) → ok

── Step 3 ──
  → read_chunk({'chunk_id': '07-chunking-strategies.md:1'}) → ok
  ◆ FINAL: According to the corpus, chunks typically have...

  Citations: 2
  Read chunks: ['07-chunking-strategies.md:0', '07-chunking-strategies.md:1']
  Answer (first 300 chars): According to the corpus, chunks typically have
  about 10-20% overlap, which means a 200-token chunk would share roughly
  20-40 tokens with each adjacent chunk. The reason is that information
  sitting at a chunk boundary is otherwise invisible to retrieval...
```

The agent's first read was a relevant-but-not-targeted chunk; it read
a second to get the specific overlap answer. This is the
**read-and-refine** pattern: the snippet looked good, the chunk
turned out to be partially relevant, so it read more. Two citations
because both reads contributed to the answer.

In [ ]:
# Failure mode 3: REPEATED-ACTION REFUSAL
# Simulate the model trying to re-search with identical arguments.

print("─" * 70)
print("FAILURE MODE 3: Repeated-action detection")
print("─" * 70)
print("Simulated trajectory: agent calls search_corpus twice with")
print("identical args. The second call is REFUSED.\n")

seen: set[str] = set()
for attempt in range(3):
    args = {"query": "chunking", "top_k": 5}
    ah = _action_hash("search_corpus", args)
    if ah in seen:
        print(f"  Attempt {attempt+1}: REFUSED (repeated_action)")
    else:
        seen.add(ah)
        print(f"  Attempt {attempt+1}: would execute (new action)")

print()
print("The mechanism is at the (tool_name, args) level. Different args =")
print("different action; same args twice = refused. This prevents the")
print("classic 'model didn't understand why the result was empty so it")
print("re-issued the same query' infinite-loop bug.")


**Sample output:**

```
──────────────────────────────────────────────────────────────────────
FAILURE MODE 3: Repeated-action detection
──────────────────────────────────────────────────────────────────────
Simulated trajectory: agent calls search_corpus twice with
identical args. The second call is REFUSED.

  Attempt 1: would execute (new action)
  Attempt 2: REFUSED (repeated_action)
  Attempt 3: REFUSED (repeated_action)

The mechanism is at the (tool_name, args) level. Different args =
different action; same args twice = refused. This prevents the
classic 'model didn't understand why the result was empty so it
re-issued the same query' infinite-loop bug.
```

Note this protection isn't subtle, but it's effective. The most
common multi-step-agent failure is re-issuing identical queries
expecting different results.

## Step 8: Citation inspection — the loop is the source of truth

Verify that the citations from earlier queries are exactly the chunks
the agent *read*, not just chunks it *saw in snippets*.

In [ ]:
# Inspect citations from the medium-difficulty query
print("─" * 70)
print("Citations from the medium query:")
print("─" * 70)
for i, c in enumerate(result_med.get("citations", []), 1):
    print(f"  {i}. [{c['chunk_id']}] {c['title']}")
print()
print(f"Total chunks READ (via read_chunk): {len(result_med.get('citations', []))}")
print()
print("The URLs above are EXACTLY the chunks the loop recorded when")
print("read_chunk returned status='ok'. The agent CANNOT have cited a")
print("chunk it didn't actually read. The model's final answer is free")
print("to phrase the synthesis however it wants — but the chunks it can")
print("ground claims in are constrained structurally by what's in this list.")


## Step 9 (stretch): Swap to OpenAI embeddings

`all-MiniLM-L6-v2` is the right default for a community lab —
no API key, runs offline. For a real project where embedding quality
matters, `text-embedding-3-small` from OpenAI is the natural swap:
1536-dim, hosted, ~$0.02 per 1M tokens.

The swap is ~15 lines. Same loop, same tools, same agent code —
only the embedding function changes.

```python
def encode_with_openai(texts: list[str]) -> np.ndarray:
    """Encode via OpenAI text-embedding-3-small. Returns (N, 1536) array."""
    from openai import OpenAI
    client = OpenAI()
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )
    arr = np.array([d.embedding for d in response.data], dtype=np.float32)
    # OpenAI's embeddings are NOT pre-normalized; normalize for dot=cosine
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / norms


# Then in step 2:
embeddings = encode_with_openai(chunk_texts)
# Different shape — (n_chunks, 1536) instead of (n_chunks, 384)
```

A few notes on the swap:

- **Re-embed the entire corpus** when you change models. Old MiniLM
  vectors can't be compared to new OpenAI vectors — they live in
  different vector spaces.
- **Storage cost is 4× higher** (1536 vs 384 dims). For the lab's 24
  chunks: 144 KB instead of 36 KB. Not relevant at this scale; very
  relevant at millions of chunks.
- **Network latency** matters for query-time embedding (every search
  is now a remote API call). For indexing it doesn't — that's a
  one-time cost.
- **Matryoshka reduction**: pass `dimensions=512` to the
  `embeddings.create` call to get a 512-dim vector — quality drops
  modestly, storage is 3× smaller. This is a unique feature of the
  `text-embedding-3-*` models.

For a production system where retrieval quality drives revenue, the
OpenAI swap is usually worth it. For learning the pattern, MiniLM is
correct.

## ✓ Lab complete

You've built an agentic RAG system end-to-end:

- Loaded and chunked a Markdown corpus with recursive structure-aware
  splitting.
- Embedded the chunks with `all-MiniLM-L6-v2` (384-dim, CPU, ~80 MB).
- Built a numpy vector index — 24 chunks × 384 dims = 36 KB. Real,
  working, brute-force-but-correct.
- Wired retrieval as two agent tools (`search_corpus` + `read_chunk`)
  mirroring Lab 03's `web_search` + `fetch_page`.
- Ran the same agent loop as Labs 01/03, with chunk-level citation
  tracking.
- Exercised three failure modes: empty results, similar-but-wrong
  retrieval, repeated-action refusal.

The pattern transfers straight from Lab 03:

| Pattern | Lab 03 (web) | Lab 06 (corpus) |
|---|---|---|
| Two tools (broad + targeted) | ✓ `web_search` + `fetch_page` | ✓ `search_corpus` + `read_chunk` |
| Structured tool errors | ✓ | ✓ |
| Citations tracked by the loop | ✓ URLs | ✓ Chunk IDs |
| Repeated-action detection | ✓ | ✓ |
| Step cap with graceful exit | ✓ | ✓ |
| Provider-agnostic LLM client | ✓ | ✓ |

### What to do next

- 🧠 **Take the quiz:** [`quizzes/agentic-rag/rag-fundamentals.md`](../../quizzes/agentic-rag/rag-fundamentals.md)
- 🧭 **Optional: extend this lab.** A few directions you might explore
  on your own:
  1. **Pluggable embedding backends.** Implement the OpenAI swap from
     step 9 and benchmark it against MiniLM on a few hard queries.
  2. **Metadata-filtered retrieval.** Add a `category` field to each
     chunk and let `search_corpus` accept a `filter` argument that
     restricts the similarity search to a subset.
  3. **MMR diversification.** When `top_k=5` returns 5 chunks from the
     same document, you've over-retrieved one source. Maximal
     Marginal Relevance picks chunks that are both query-relevant and
     mutually diverse. ~20 lines on top of the current code.
  4. **Persistent index.** Save `embeddings` and `all_chunks` to disk
     so re-runs don't re-embed. `np.save(...)` + JSON for metadata.
- 🧭 **Continue Path 02** with the next batch (when it lands): retrieval
  strategies, re-ranking, hybrid search, contextual retrieval.
- 🧭 **Or move on to Path 03 (Multi-Agent Systems) or Path 06
  (Evaluation & Observability).** Both paths build on what you've
  learned here.